# ModuleNotFoundError & sys.path Explained


<!-- © Gemini | AI generated with my own alterations -->

# Introduction 

Actual python that caused the error:

```py
from src.core import JSON_DATA_FILE_PATH
import json


class ProductRepository:
    # 1. get all products
    def get_all(self) -> dict | None:

        payload = None

        if JSON_DATA_FILE_PATH:
            # read json data
            with open(JSON_DATA_FILE_PATH, mode="r") as file:
                payload = json.load(file)

            # DEBUG: inspect few values
            # print("--- DEBUG Inspect payload ---")
            # print(payload[1:5])
            # print("-"*25)

            return {"total_products": len(payload), "products": payload}
        else:
            return None


# Debugging
product = ProductRepository()

# Get all products
print("--- DEBUG product repository ---")
all_products: dict | None = product.get_all()
if all_products:
    # print some products
    print(all_products["products"][1:2])
```

Console output:
```bash
saqibbedar@bedar:~/dev/ai-ml/main/ai-ml$ uv run python fastapi-projects/4-project/src/repositories/product.py 
Traceback (most recent call last):
  File "/home/saqibbedar/dev/ai-ml/main/ai-ml/fastapi-projects/4-project/src/repositories/product.py", line 1, in <module>
    from src.core import JSON_DATA_FILE_PATH
ModuleNotFoundError: No module named 'src'
```

---

### Let's trace your exact terminal command:

You ran:
```bash
saqibbedar@bedar:~/dev/ai-ml/main/ai-ml$ uv run python fastapi-projects/4-project/src/repositories/product.py
```

Here is what Python did behind the scenes:
1. Python looked at the file you gave it: `.../4-project/src/repositories/product.py`.
2. Python set `sys.path[0]` to:
   ```text
   /home/saqibbedar/dev/ai-ml/main/ai-ml/fastapi-projects/4-project/src/repositories/
   ```
   *(Notice: it is inside `repositories/`, not `4-project/`!)*
3. Line 1 of `product.py` ran:
   ```python
   from src.core import JSON_DATA_FILE_PATH
   ```
4. Python looked inside `sys.path[0]` (`.../src/repositories/`):  
   *"Is there a directory named `src` inside `repositories/`?"*  
   **No!** `src` is two levels above `repositories/`.
5. Python searched the standard library and site-packages: **No `src` there either.**
6. **Result:** `ModuleNotFoundError: No module named 'src'`.

---

# Solution

[Read solution](solution.md) 

### 3. How to Run a Specific File in Python

When you want to run a specific nested file for quick testing, here are the standard ways to handle it:

#### Method 1: The `-m` (Module) Flag (The Standard Python CLI Way)
Instead of passing the file path, pass the **module dot-notation** using the `-m` flag from the project root:

```bash
cd ~/dev/ai-ml/main/ai-ml/fastapi-projects/4-project
uv run python -m src.repositories.product
```

**Why this works:**
The `-m` flag tells Python:  
*"Set `sys.path[0]` to my **current terminal directory** (`4-project`), NOT the subfolder `repositories`. Then find and execute `src.repositories.product`."*  
Now, because `4-project` is in `sys.path[0]`, `from src.core ...` finds `src/` instantly!

---

#### Method 2: Tell Python where the root is via `PYTHONPATH`
If you don't want to `cd` into the project and want to run it from your top-level workspace:

```bash
PYTHONPATH=fastapi-projects/4-project uv run python fastapi-projects/4-project/src/repositories/product.py
```

**Why this works:**
`PYTHONPATH` forces Python to prepend `fastapi-projects/4-project` into `sys.path`. Now Python can find `src` regardless of which nested file you invoked.

---

#### Method 3: Relative Import vs. Absolute Import
Inside `src/repositories/product.py`, if you wanted to import `core` relatively without relying on `src` at the root:

```python
from ..core import JSON_DATA_FILE_PATH
```
*(The two dots `..` mean "go up one level from `repositories` to `src`, then into `core`").*  
However, in Python, relative imports **only work when executed as a package/module** (e.g. via `uvicorn` or `python -m`), never when directly executed as a standalone script file (`python file.py`).

---

### Summary Checklist

| If you run this: | `sys.path[0]` becomes: | Will `from src...` work? |
| :--- | :--- | :--- |
| `python src/repositories/product.py` | `src/repositories/` | ❌ **No** (looks for `repositories/src`) |
| `python -m src.repositories.product` *(from project root)* | Project root (`4-project/`) | ✅ **Yes** (looks for `4-project/src`) |
| `PYTHONPATH=. python src/repositories/product.py` | Current directory + `PYTHONPATH` | ✅ **Yes** (finds `src` in search path) |
| Running with `uvicorn src.main:app` | Project root | ✅ **Yes** (Uvicorn adds project root to `sys.path`) |
